# **MODEL TRAINING**
##We will use **NLP** and **Cosine similarity** for the Model Training



### 1.Import all required libraries

In [3]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

###2.Load the Cleaned Dataset

In [4]:
df = pd.read_csv('../Data/cleaned_movies.csv')

###3.Import NLP Library

In [5]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\satvi\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\satvi\AppData\Roaming\nltk_data...


True

In [6]:
stop_words=set(stopwords.words('english'))
lemmatizer=WordNetLemmatizer()


###4.Tokenization,Stemming,Stopwords handling

In [7]:
def text_clean(text):
  #lowercase
    text=text.lower()

  #remove punctuations
    text=re.sub('[^a-zA-Z]',' ',text)

    words=text.split()
  #remove stopwords
    words=[word for word in words if word not in stop_words]

  #lemmatize
    words=[lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)


In [8]:
df['description']=df['description'].apply(text_clean)

In [9]:
df=df.reset_index(drop=True)

In [10]:
indices=pd.Series(df.index,index=df['title']).drop_duplicates()

In [11]:
indices

title
Toy Story                          0
Jumanji                            1
Grumpier Old Men                   2
Waiting to Exhale                  3
Father of the Bride Part II        4
                               ...  
Subdue                         45442
Century of Birthing            45443
Betrayal                       45444
Satan Triumphant               45445
Queerama                       45446
Length: 45447, dtype: int64

###5.Importing TF-IDF

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [13]:
tfidf = TfidfVectorizer(max_features=50000,ngram_range=(1,2),stop_words='english')

In [14]:
tfidf_matrix = tfidf.fit_transform(df['description'])

In [15]:
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
def recommend(title, n = 5):
  if title not in indices:
    return ['Movie not found']

  idx = indices[title]
  sim_score = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
  similar_idx = sim_score.argsort()[::-1][1:n+1]
  return df['title'].iloc[similar_idx]

###6.Exporting the .pkl file of the trained model

In [17]:
import pickle

pickle.dump(tfidf_matrix,open('tfidf_matrix.pkl','wb'))

pickle.dump(indices,open('indices.pkl','wb'))

df.to_pickle('df.pkl')

pickle.dump(tfidf,open('tfidf.pkl','wb'))